<a href="https://colab.research.google.com/github/dakshesh-1293/song-recommendation-system/blob/main/spotify_song_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# ML-BASED SPOTIFY SONG RECOMMENDATION SYSTEM
# ============================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('/content/drive/MyDrive/Machine Learning /Data Sets/spotify.csv')
print("="*50)
print("ML-POWERED SONG RECOMMENDER")
print("="*50)

# ============================================
# DATA PREPROCESSING (ML Step 1)
# ============================================

# Select features for ML
ml_features = ['danceability', 'energy', 'loudness', 'instrumentalness', 'tempo', 'popularity']

# Clean data
df_clean = df.dropna(subset=['track_name', 'artist_name', 'genre'])
for col in ml_features:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Scale features (ML Step 2)
scaler = StandardScaler()
df_clean[ml_features] = scaler.fit_transform(df_clean[ml_features])

# ============================================
# TRAIN ML MODELS (ML Step 3)

# ============================================

# 1. K-Means Clustering Model
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(df_clean[ml_features])

# 2. Nearest Neighbors Model
knn = NearestNeighbors(n_neighbors=11, metric='cosine')
knn.fit(df_clean[ml_features])

print("✅ ML models trained successfully!")
print(f"   - {len(df_clean)} songs processed")
print(f"   - {len(ml_features)} features used")
print(f"   - 8 song clusters created")

# ============================================
# ML-BASED RECOMMENDATION FUNCTIONS
# ============================================

def recommend_by_song_ml(song_name, n_recommendations=5):
    """ML-based recommendation using cosine similarity"""

    # Find the song
    song = df_clean[df_clean['track_name'].str.lower() == song_name.lower()]
    if len(song) == 0:
        song = df_clean[df_clean['track_name'].str.lower().str.contains(song_name.lower())]

    if len(song) == 0:
        return None

    song = song.iloc[0]

    # Get similar songs using cosine similarity on scaled features
    song_features = song[ml_features].values.reshape(1, -1)
    similarities = cosine_similarity(song_features, df_clean[ml_features])[0]

    # Get top similar songs (excluding itself)
    similar_indices = similarities.argsort()[::-1][1:n_recommendations+1]

    recommendations = []
    for idx in similar_indices:
        rec = df_clean.iloc[idx]  #👉 Fetch song details
        recommendations.append({
            'track_name': rec['track_name'],
            'artist_name': rec['artist_name'],
            'genre': rec['genre'],
            'similarity': similarities[idx],
            'cluster': rec['cluster']
        })

    return recommendations, song


def recommend_by_cluster_ml(song_name, n_recommendations=5):
    """ML-based recommendation using K-Means clustering"""

    # Find the song

    #👉 Find exact match
    song = df_clean[df_clean['track_name'].str.lower() == song_name.lower()]

    #👉 Try partial match
    if len(song) == 0:
        song = df_clean[df_clean['track_name'].str.lower().str.contains(song_name.lower())]

    #👉 Stop function
    if len(song) == 0:
        return None

    song = song.iloc[0]  #👉 If multiple matches → pick first

    # Get all songs from same cluster
    #Find all songs with same cluster number
    same_cluster = df_clean[df_clean['cluster'] == song['cluster']]

    # Exclude the input song and get top by popularity
    same_cluster = same_cluster[same_cluster['track_name'] != song['track_name']]
    recommendations = same_cluster.nlargest(n_recommendations, 'popularity')

    return recommendations[['track_name', 'artist_name', 'genre', 'popularity']], song


def recommend_by_knn_ml(song_name, n_recommendations=5):
    """ML-based recommendation using K-Nearest Neighbors"""

    # Find the song
    song = df_clean[df_clean['track_name'].str.lower() == song_name.lower()]
    if len(song) == 0:
        song = df_clean[df_clean['track_name'].str.lower().str.contains(song_name.lower())]

    if len(song) == 0:
        return None

    idx = song.index[0]  #👉 Get position of song in dataset
    song_data = song[ml_features].values.reshape(1, -1)

    # Find nearest neighbors
    distances, indices = knn.kneighbors(song_data, n_neighbors=n_recommendations+1)

    recommendations = []
    for i in range(1, len(indices[0])):
        rec_idx = indices[0][i]
        rec = df_clean.iloc[rec_idx]
        recommendations.append({
            'track_name': rec['track_name'],
            'artist_name': rec['artist_name'],
            'genre': rec['genre'],
            'distance': distances[0][i]
        })

    return recommendations, song.iloc[0]


def recommend_by_genre_ml(genre, n_recommendations=10):
    """ML-enhanced genre recommendations (using popularity)"""

    genre_songs = df_clean[df_clean['genre'].str.lower() == genre.lower()]
    if len(genre_songs) == 0:
        genre_songs = df_clean[df_clean['genre'].str.lower().str.contains(genre.lower())]

    if len(genre_songs) == 0:
        return None

    # Sort by popularity (ML feature)
    return genre_songs.nlargest(n_recommendations, 'popularity')


def recommend_by_artist_ml(artist, n_recommendations=10):
    """ML-enhanced artist recommendations"""

    artist_songs = df_clean[df_clean['artist_name'].str.lower() == artist.lower()]
    if len(artist_songs) == 0:
        artist_songs = df_clean[df_clean['artist_name'].str.lower().str.contains(artist.lower())]

    if len(artist_songs) == 0:
        return None

    # Sort by popularity
    return artist_songs.nlargest(n_recommendations, 'popularity')




# ============================================
# INTERACTIVE MENU
# ============================================

while True:
    print("\n" + "="*50)
    print("🤖 ML-POWERED RECOMMENDATION MENU")
    print("="*50)
    print("\n1. Cosine Similarity (Content-based ML)")
    print("2. K-Means Clustering (Unsupervised ML)")
    print("3. K-Nearest Neighbors (Distance-based ML)")
    print("4. Genre Recommendations (ML-enhanced)")
    print("5. Artist Recommendations (ML-enhanced)")
    print("6. Exit")

    choice = input("\nYour choice (1-6): ").strip()

    if choice == '1':
        # Cosine Similarity ML
        song_name = input("\nEnter song name: ").strip()
        n = input("Number of recommendations (default 5): ").strip()
        n = int(n) if n.isdigit() else 5

        result = recommend_by_song_ml(song_name, n)

        if result is None:
            print(f"\n❌ Song '{song_name}' not found!")
        else:
            recommendations, original_song = result
            print(f"\n✅ ML Recommendation for: {original_song['track_name']} by {original_song['artist_name']}")
            print(f"   Genre: {original_song['genre']} | Cluster: {original_song['cluster']}\n")

            for i, rec in enumerate(recommendations, 1):
                print(f"{i}. {rec['track_name']}")
                print(f"   Artist: {rec['artist_name']}")
                print(f"   Genre: {rec['genre']}")
                print(f"   ML Similarity Score: {rec['similarity']:.3f}")
                print(f"   Cluster: {rec['cluster']}")
                print()

    elif choice == '2':
        # K-Means Clustering ML
        song_name = input("\nEnter song name: ").strip()
        n = input("Number of recommendations (default 5): ").strip()
        n = int(n) if n.isdigit() else 5

        result = recommend_by_cluster_ml(song_name, n)

        if result is None:
            print(f"\n❌ Song '{song_name}' not found!")
        else:
            recommendations, original_song = result
            print(f"\n✅ ML Cluster Recommendation for: {original_song['track_name']}")
            print(f"   Cluster ID: {original_song['cluster']}")
            print(f"   Genre: {original_song['genre']}\n")
            print(f"   Other songs in same cluster:\n")

            for i, (idx, rec) in enumerate(recommendations.iterrows(), 1):
                print(f"{i}. {rec['track_name']}")
                print(f"   Artist: {rec['artist_name']}")
                print(f"   Genre: {rec['genre']}")
                print(f"   Popularity Score: {rec['popularity']:.2f}")
                print()

    elif choice == '3':
        # KNN ML
        song_name = input("\nEnter song name: ").strip()
        n = input("Number of recommendations (default 5): ").strip()
        n = int(n) if n.isdigit() else 5

        result = recommend_by_knn_ml(song_name, n)

        if result is None:
            print(f"\n❌ Song '{song_name}' not found!")
        else:
            recommendations, original_song = result
            print(f"\n✅ KNN ML Recommendation for: {original_song['track_name']}")
            print(f"   Artist: {original_song['artist_name']}")
            print(f"   Genre: {original_song['genre']}\n")

            for i, rec in enumerate(recommendations, 1):
                print(f"{i}. {rec['track_name']}")
                print(f"   Artist: {rec['artist_name']}")
                print(f"   Genre: {rec['genre']}")
                print(f"   ML Distance: {rec['distance']:.4f}")
                print()

    elif choice == '4':
        # Genre ML
        print("\nTop Genres in Dataset:")
        genres = df_clean['genre'].value_counts().head(10)
        for i, (genre, count) in enumerate(genres.items(), 1):
            print(f"   {i}. {genre} ({count} songs)")

        genre_name = input("\nEnter genre: ").strip()
        n = input("How many songs? (default 10): ").strip()
        n = int(n) if n.isdigit() else 10

        result = recommend_by_genre_ml(genre_name, n)

        if result is None:
            print(f"\n❌ Genre '{genre_name}' not found!")
        else:
            print(f"\n✅ Top {genre_name} songs (ML-ranked by popularity):\n")
            for i, (idx, song) in enumerate(result.iterrows(), 1):
                print(f"{i}. {song['track_name']}")
                print(f"   Artist: {song['artist_name']}")
                print(f"   ML Popularity Score: {song['popularity']:.2f}")
                print()

    elif choice == '5':
        # Artist ML
        artist_name = input("\nEnter artist name: ").strip()
        n = input("How many songs? (default 10): ").strip()
        n = int(n) if n.isdigit() else 10

        result = recommend_by_artist_ml(artist_name, n)

        if result is None:
            print(f"\n❌ Artist '{artist_name}' not found!")
        else:
            print(f"\n✅ Top songs by {artist_name} (ML-ranked):\n")
            for i, (idx, song) in enumerate(result.iterrows(), 1):
                print(f"{i}. {song['track_name']}")
                print(f"   Genre: {song['genre']}")
                print(f"   ML Popularity Score: {song['popularity']:.2f}")
                print()

    elif choice == '6':
        print("\n" + "="*50)
        print("👋 Thanks for using ML-Powered Song Recommender!")
        print("="*50)
        break

    else:
        print("\n❌ Invalid choice! Please enter 1-7.")

    input("\nPress Enter to continue...")

ML-POWERED SONG RECOMMENDER
✅ ML models trained successfully!
   - 84979 songs processed
   - 6 features used
   - 8 song clusters created

🤖 ML-POWERED RECOMMENDATION MENU

1. Cosine Similarity (Content-based ML)
2. K-Means Clustering (Unsupervised ML)
3. K-Nearest Neighbors (Distance-based ML)
4. Genre Recommendations (ML-enhanced)
5. Artist Recommendations (ML-enhanced)
6. Exit

Your choice (1-6): 2

Enter song name: Together
Number of recommendations (default 5): 3

✅ ML Cluster Recommendation for: Together
   Cluster ID: 1
   Genre: Folk

   Other songs in same cluster:

1. Role
   Artist: Anthony Gonzalez
   Genre: Indie
   Popularity Score: 2.35

2. Radio learn kind east
   Artist: Michele Contreras
   Genre: EDM
   Popularity Score: 2.08

3. Offer
   Artist: Marissa Navarro
   Genre: Rock
   Popularity Score: 2.01


Press Enter to continue...

🤖 ML-POWERED RECOMMENDATION MENU

1. Cosine Similarity (Content-based ML)
2. K-Means Clustering (Unsupervised ML)
3. K-Nearest Neighbors